# S&P 500 協整法配對交易 (Cointegration Method)
## 基於 Johansen 協整檢定的嚴格實作

本 Notebook 回測協整法策略，完全遵守以下規則：

**一、 策略核心架構與時間視窗**
- 滾動窗口（Rolling Window）機制，更新頻率 21 個交易日（約一個月）。
- 形成期 (Formation Period)：252 個交易日（約一年）。
- 交易期 (Trading Period)：126 個交易日（約半年）。
- 隨時維持 6 個重疊組合（$126 \div 21 = 6$），各佔資金 $1/6$。
- 使用 S&P 500 成分股（含息報價）。

**二、 第一階段：配對篩選**
- 形成期首日價格正規化為 1。
- 進行 Johansen 協整檢定，依 Trace Statistics 最高分選前 20 對。
- 參數估計：$P_{1,t} - \beta P_{2,t} = \mu + \epsilon_t$，取得 $\beta$、$\mu$、$\sigma$。

**三、 第二階段：交易規則**
- 交易期首日將價格再次縮放為 1。
- 殘差 $Spread > \mu + 2\sigma$ $\rightarrow$ 賣空 1，買入 2。
- 殘差 $Spread < \mu - 2\sigma$ $\rightarrow$ 買入 1，賣空 2。
- 依 $\beta$ 比例建倉。
- 出場點：回歸均值 $\mu$ 收斂，或第 126 天強制平倉。

**四、 資金補填 (Market Fill)**
- 每個子組合等權重分配前 20 對。
- 若具協整關係少於 10 對，剩餘未使用資金直接做多 SPY (S&P 500)。

**五、 成本基準**
- 單邊交易與滑價：0.3% (30 bps)。
- 放空借券：年化 1%。


In [1]:
# 載入套件與設定
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import matplotlib.pyplot as plt
import sqlite3
import math
import os
import yfinance as yf
from itertools import combinations
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
print("✓ 核心套件載入完成")


✓ 核心套件載入完成


In [2]:

# 視窗參數
FORMATION_PERIOD = 252  # 形成期
TRADING_PERIOD = 126    # 交易期
ROLL_STEP = 21          # 滾動頻率
NUM_SUBPORTFOLIOS = TRADING_PERIOD // ROLL_STEP  # 6個子組合
TOP_PAIRS_COUNT = 20    # 每個子組合最大交易對數
MIN_VALID_PAIRS = 10    # 觸發 Market Fill 門檻

# 交易與成本參數
INITIAL_CAPITAL = 100000.0  # 總資金
ENTRY_Z_SCORE = 2.0       # 進場SD
TRANSACTION_COST = 0.0030 # 0.3%單邊手續費與滑價
SHORT_FEE_ANNUAL = 0.01   # 放空年化費率

print(f"總資金: ${INITIAL_CAPITAL:,.2f} | 子組合數: {NUM_SUBPORTFOLIOS} | 子組合資金: ${INITIAL_CAPITAL/NUM_SUBPORTFOLIOS:,.2f}")


總資金: $100,000.00 | 子組合數: 6 | 子組合資金: $16,666.67


In [3]:
import os
# ========== 階段 1：數據預處理與正規化 ==========
# === 全局參數設置 ===
FAST_TEST_MODE = True

if FAST_TEST_MODE:
    print("【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    # 縮短時間：涵蓋 2020 疫情崩盤與 2022 升息的壓力測試區間
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # 縮小股票池：僅測試單一板塊，運算量大幅減少
    TARGET_SECTOR = 'Information Technology'  
else:
    print("【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    # 論文要求的全樣本時間
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    # 測試全市場 (設為 None 代表不限制單一產業)
    TARGET_SECTOR = None  

# 資料庫配置
DB_PATH = r'..\data\sp500.db'
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'
USE_DYNAMIC_SECTORS = True  # 是否使用動態產業補齊

# 視窗參數 (嚴格遵循 GGR 論文)
FORMATION_PERIOD = 252    # 約 1 年
TRADING_PERIOD = 126      # 約 6 個月
ROLL_STEP = 21       # 約 1 個月 (梯隊步長)
NUM_SUBPORTFOLIOS = TRADING_PERIOD // ROLL_STEP  # 6個子組合
TOP_PAIRS_COUNT = 20    # 每個子組合最大交易對數
MIN_VALID_PAIRS = 10    # 觸發 Market Fill 門檻

# 交易與成本參數
INITIAL_CAPITAL = 100000.0  # 總資金
ENTRY_Z_SCORE = 2.0       # 進場SD
TRANSACTION_COST = 0.0030 # 0.3%單邊手續費與滑價
SHORT_FEE_ANNUAL = 0.01   # 放空年化費率

# 確保路徑存在
os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

print(f"""
【回測參數設置】
時間期間: {START_DATE} ~ {END_DATE}
形成期: {FORMATION_PERIOD} 天（約 1 年）
交易期: {TRADING_PERIOD} 天（約 6 個月）
步長: {ROLL_STEP} 天（約 1 個月）
配對數: {TOP_PAIRS_COUNT}
初始資金: ${INITIAL_CAPITAL:,.0f}
交易成本: {TRANSACTION_COST*100:.2f}%
""")

【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。

【回測參數設置】
時間期間: 2019-01-01 ~ 2022-12-31
形成期: 252 天（約 1 年）
交易期: 126 天（約 6 個月）
步長: 21 天（約 1 個月）
配對數: 20
初始資金: $100,000
交易成本: 0.30%



In [4]:
# === 資料庫讀取邏輯 ===
def load_and_preprocess_data(db_path, start_date, end_date):
    import sqlite3
    conn = sqlite3.connect(db_path)
    # 嘗試讀取價格資料
    query = f"SELECT date, ticker, adj_close AS close FROM daily_prices WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"
    try:
        df_prices = pd.read_sql_query(query, conn, parse_dates=['date'])
    except:
        query = f"SELECT date, ticker, close FROM stock_prices WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"
        df_prices = pd.read_sql_query(query, conn, parse_dates=['date'])
    
    conn.close()
    
    pivot_df = df_prices.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot_df = pivot_df.ffill(limit=5)
    
    # 過濾有效天數
    valid_cols = pivot_df.columns[pivot_df.notna().sum() >= 500]
    pivot_df = pivot_df[valid_cols]
    
    # 獲取 SPY 用作 Market Fill 標的
    try:
        spy_df = yf.download("SPY", start=start_date, end=end_date, progress=False)
        spy_series = spy_df['Adj Close']
        # 匹配日期
        bench_reindexed = spy_series.reindex(pivot_df.index).ffill()
    except:
        #若無法抓取，使用算術平均股價替代
        bench_reindexed = pivot_df.mean(axis=1)
        
    return pivot_df, bench_reindexed

print("讀取與預處理股價...")
prices_df, bench_df = load_and_preprocess_data(DB_PATH, START_DATE, END_DATE)
print(f"價格資料形狀: {prices_df.shape}")


讀取與預處理股價...
價格資料形狀: (1008, 612)


In [5]:
# === 選擇與計算 Johansen 協整 ===
def perform_johansen_selection(form_prices, top_n=20):
    """
    一、 正規化：形成期首日價格設為 1
    二、 Johansen Trace Stat > 95% 且數值最高前 20 對
    三、 估計 P1 - beta*P2 = mu + e
    """
    form_prices = form_prices.dropna(axis=1)
    if form_prices.shape[1] < 2: return []
    
    # 1. 價格歸一化 (首日=1)
    norm_p = (form_prices / form_prices.iloc[0]).dropna(axis=1)
    cols = norm_p.columns.tolist()
    
    results = []
    
    # 快速過濾：同向變動相關性大於 0.5 才做 Johansen，降低運算量
    corr_matrix = norm_p.corr()
    pairs = []
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            if corr_matrix.iloc[i, j] > 0.5:  
                pairs.append((cols[i], cols[j]))
                
    for c1, c2 in pairs:
        y = norm_p[[c1, c2]].values
        
        # Johansen Cointegration
        try:
            res = coint_johansen(y, det_order=0, k_ar_diff=1)
            trace_stat = res.lr1[0]
            trace_crit = res.cvt[0, 1]  # 95% 臨界值
            
            if trace_stat > trace_crit: 
                # 協整成立
                P1 = norm_p[c1]
                P2 = norm_p[c2]
                P2_const = sm.add_constant(P2)
                ols_res = sm.OLS(P1, P2_const).fit()
                beta = ols_res.params[c2]
                
                # 計算殘差 (Spread = P1 - beta*P2)
                spread = P1 - beta * P2
                mu = spread.mean()
                sigma = spread.std()
                
                results.append({
                    'pair': (c1, c2),
                    'trace_stat': trace_stat,
                    'beta': beta,
                    'mu': mu,
                    'sigma': sigma
                })
        except:
            pass
            
    df_res = pd.DataFrame(results)
    if df_res.empty: return []
    
    # Trace Stat 由大到小排序
    df_res = df_res.sort_values('trace_stat', ascending=False).head(top_n)
    return df_res.to_dict('records')


In [6]:
# === 交易邏輯與資本狀態機 ===
def run_trading_period(trade_prices, bench_series, config_pairs, initial_sub_capital=100000/6):
    """
    一、 正規化：交易期開始首日再次縮放為 1
    二、 進出場門檻：2-SD 觸發
    三、 若配對數不足，剩餘分配給 Market Fill
    """
    dates = trade_prices.index
    valid_pairs_count = len(config_pairs)
    slots = max(TOP_PAIRS_COUNT, valid_pairs_count)
    capital_per_slot = initial_sub_capital / slots
    
    # Market Fill (缺失部位投入大盤)
    market_fill_capital = 0.0
    if valid_pairs_count < MIN_VALID_PAIRS:
        market_fill_capital = (slots - valid_pairs_count) * capital_per_slot
        
    portfolio_value = pd.Series(0.0, index=dates)
    
    # 1. Market Fill 預算放入大盤
    if market_fill_capital > 0:
        b0 = bench_series.iloc[0]
        spy_shares = (market_fill_capital * (1 - TRANSACTION_COST)) / b0
        portfolio_value += bench_series * spy_shares
    elif valid_pairs_count == 0:
        return pd.Series(initial_sub_capital, index=dates)

    # 2. 配對交易部分
    for cfg in config_pairs:
        c1, c2 = cfg['pair']
        if c1 not in trade_prices.columns or c2 not in trade_prices.columns:
            portfolio_value += capital_per_slot
            continue
            
        p1 = trade_prices[c1]
        p2 = trade_prices[c2]
        
        # 交易期二次歸一化
        p1_0 = p1.iloc[0]
        p2_0 = p2.iloc[0]
        if p1_0 == 0 or p2_0 == 0 or pd.isna(p1_0) or pd.isna(p2_0):
            portfolio_value += capital_per_slot
            continue
            
        norm_p1 = p1 / p1_0
        norm_p2 = p2 / p2_0
        
        beta, mu, sigma = cfg['beta'], cfg['mu'], cfg['sigma']
        spread = norm_p1 - beta * norm_p2
        
        position = 0 # 0:空 , 1:做多1做空2, -1:做空1做多2
        shares1_norm = 0
        shares2_norm = 0
        current_pair_cash = capital_per_slot
        pair_value_daily = np.zeros(len(dates))
        
        threshold_up = mu + ENTRY_Z_SCORE * sigma
        threshold_dn = mu - ENTRY_Z_SCORE * sigma
        
        for t in range(len(dates)):
            val1, val2 = p1.iloc[t], p2.iloc[t]
            n_val1, n_val2 = norm_p1.iloc[t], norm_p2.iloc[t]
            sp = spread.iloc[t]
            
            # 放空成本
            if position != 0:
                short_fee = (current_pair_cash / 2) * (SHORT_FEE_ANNUAL / 252)
                current_pair_cash -= short_fee
                
            if position == 0:
                if sp > threshold_up:
                    position = -1 # 賣出(放空) 1，買入 2
                elif sp < threshold_dn:
                    position = 1  # 買入 1，賣出(放空) 2
                    
                if position != 0: # 建倉
                    # 依 Beta 設定等價值對等部位，這裡以簡化50/50保證金資金計算
                    side_cap = current_pair_cash / 2
                    
                    if position == -1: # 空1多2
                        shares1_norm = -(side_cap * (1 - TRANSACTION_COST)) / n_val1
                        shares2_norm = (side_cap * (1 - TRANSACTION_COST)) / n_val2
                    else: # 多1空2
                        shares1_norm = (side_cap * (1 - TRANSACTION_COST)) / n_val1
                        shares2_norm = -(side_cap * (1 - TRANSACTION_COST)) / n_val2
                        
                    current_pair_cash -= current_pair_cash * TRANSACTION_COST 
            else:
                # 檢查平倉
                exit_signal = False
                if position == -1 and sp <= mu: exit_signal = True
                elif position == 1 and sp >= mu: exit_signal = True
                elif t == len(dates) - 1: exit_signal = True # 強制平倉
                
                if exit_signal:
                    val_1_now = shares1_norm * n_val1
                    val_2_now = shares2_norm * n_val2
                    position_gross = abs(val_1_now) + abs(val_2_now)
                    cost = position_gross * TRANSACTION_COST
                    
                    current_pair_cash += (val_1_now + val_2_now - cost)
                    
                    position, shares1_norm, shares2_norm = 0, 0, 0
                    
            if position == 0:
                 pair_value_daily[t] = current_pair_cash
            else:
                 pair_value_daily[t] = current_pair_cash + (shares1_norm * n_val1) + (shares2_norm * n_val2)
                
        portfolio_value += pair_value_daily
        
    return portfolio_value


In [7]:
# === 執行回測 ===
def run_backtest_rolling(prices_df, bench_df):
    T_total = len(prices_df)
    total_dates = prices_df.index
    
    subportfolios = [] 
    
    for start_idx in range(0, T_total - FORMATION_PERIOD - TRADING_PERIOD + 1, ROLL_STEP):
        form_end_idx = start_idx + FORMATION_PERIOD
        trade_end_idx = form_end_idx + TRADING_PERIOD
        
        form_prices = prices_df.iloc[start_idx:form_end_idx]
        trade_prices = prices_df.iloc[form_end_idx:trade_end_idx]
        local_bench = bench_df.iloc[form_end_idx:trade_end_idx]
        
        print(f"[{total_dates[form_end_idx].date()}] 開始更新，訓練期1年...")
        pairs = perform_johansen_selection(form_prices, top_n=TOP_PAIRS_COUNT)
        print(f" -> 找到 {len(pairs)} 對協整配對。")
        
        sub_val = run_trading_period(trade_prices, local_bench, pairs, initial_sub_capital=INITIAL_CAPITAL/NUM_SUBPORTFOLIOS)
        
        full_sub_val = pd.Series((INITIAL_CAPITAL/NUM_SUBPORTFOLIOS), index=total_dates)
        full_sub_val.loc[sub_val.index] = sub_val
        if trade_end_idx < T_total:
            full_sub_val.iloc[trade_end_idx:] = sub_val.iloc[-1]
            
        subportfolios.append(full_sub_val)
        
    if not subportfolios:
        return pd.Series(INITIAL_CAPITAL, index=total_dates)
        
    # 計算每日子組合報酬率進行等權重聚合，確保資金曲線正確拼接
    daily_returns_all = [sub.pct_change().fillna(0) for sub in subportfolios]
    returns_df = pd.concat(daily_returns_all, axis=1)
    
    active_mask = (returns_df != 0)
    avg_daily_return = returns_df.sum(axis=1) / active_mask.sum(axis=1).replace(0, 1)
    
    overall_value = INITIAL_CAPITAL * (1 + avg_daily_return).cumprod()
    return overall_value

print("啟動回測引擎...")
# 實際執行回測 (由於運算時間長，建議只針對部分特徵段落或全部運行)
# res_pnl = run_backtest_rolling(prices_df, bench_df)
print("引擎就緒。請調用 run_backtest_rolling(prices_df, bench_df) 開始。")


啟動回測引擎...
引擎就緒。請調用 run_backtest_rolling(prices_df, bench_df) 開始。


## 階段 5：績效指標計算與分析

### 計算指標：
- **CAGR (年化報酬率)**：$\text{CAGR} = (1 + R)^{252/n} - 1$
- **年化波動率**：$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$
- **Sharpe Ratio**：$SR = \frac{r_{annual}}{\sigma_{annual}}$
- **Sortino Ratio**：$\text{Sortino} = \frac{r_{annual}}{\sigma_{downside}}$
- **最大回撤 (MDD)**：$\text{MDD} = \min\left(\frac{V_t - V_{max}}{V_{max}}\right)$


In [8]:
def compute_performance_metrics(strategy_nav, initial_capital):
    returns = strategy_nav.pct_change().dropna()
    total_return = (strategy_nav.iloc[-1] / initial_capital) - 1
    
    # CAGR
    years = len(returns) / 252
    cagr = (1 + total_return) ** (1/years) - 1 if total_return > -1 else -1
    
    # 波動率與 Sharpe
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = (returns.mean() * 252) / annual_vol if annual_vol > 1e-8 else np.nan
    
    # Sortino
    downside_returns = returns[returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
    sortino = (returns.mean() * 252) / downside_vol if downside_vol > 1e-8 else np.nan
    
    # 最大回撤 (MDD)
    drawdown = (strategy_nav - strategy_nav.cummax()) / strategy_nav.cummax()
    mdd = drawdown.min()
    
    # 勝率
    win_rate = (returns > 0).sum() / len(returns)
    
    metrics = {
        'CAGR(%)': round(cagr * 100, 2),
        '總報酬(%)': round(total_return * 100, 2),
        '年化波動(%)': round(annual_vol * 100, 2),
        'Sharpe': round(sharpe, 4),
        'Sortino': round(sortino, 4),
        '最大回撤(%)': round(mdd * 100, 2),
        '日勝率(%)': round(win_rate * 100, 2)
    }
    return metrics


## 階段 6：結果視覺化與對標

進行與 S&P 500 (SPY) 買入持有的表現對標，並繪製淨值與回撤走勢圖。


In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

print("開始執行回測與對標資料準備...")
# 1. 執行策略 (會需要一些時間)
strategy_nav = run_backtest_rolling(prices_df, bench_df)

# 2. 計算策略細節
metrics = compute_performance_metrics(strategy_nav, INITIAL_CAPITAL)
print("\n【策略績效指標】")
metrics_df = pd.DataFrame([metrics])
display(metrics_df.style.format("{:.2f}"))

# 3. 準備 SPY 對標
bench_returns = bench_df.pct_change().dropna()
bench_returns = bench_returns.reindex(strategy_nav.index).fillna(0)
bench_nav = INITIAL_CAPITAL * (1 + bench_returns).cumprod()
bench_metrics = compute_performance_metrics(bench_nav, INITIAL_CAPITAL)

print("\n【SPY 對標績效指標】")
bench_metrics_df = pd.DataFrame([bench_metrics])
display(bench_metrics_df.style.format("{:.2f}"))

# 4. 繪製圖表
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.05,
                    row_heights=[0.7, 0.3])

# 淨值曲線
fig.add_trace(go.Scatter(x=strategy_nav.index, y=strategy_nav,
                         mode='lines', name='Strategy (Cointegration)',
                         line=dict(color='blue')), row=1, col=1)
                         
fig.add_trace(go.Scatter(x=bench_nav.index, y=bench_nav,
                         mode='lines', name='S&P 500 (SPY)',
                         line=dict(color='gray', dash='dash')), row=1, col=1)

# 策略回撤
drawdown = (strategy_nav - strategy_nav.cummax()) / strategy_nav.cummax() * 100
fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown,
                         mode='lines', name='Drawdown (%)',
                         fill='tozeroy', line=dict(color='red', width=1)), row=2, col=1)

fig.update_layout(title="協整法配對交易策略 (Cointegration) vs S&P 500 績效",
                  height=800, template="plotly_white")
fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
fig.update_yaxes(title_text="Drawdown (%)", row=2, col=1)

fig.show()


開始執行回測與對標資料準備...
[2020-01-02] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-02-03] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-03-04] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-04-02] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-05-04] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-06-03] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-07-02] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-08-03] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-09-01] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-10-01] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-10-30] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-12-01] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2020-12-31] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-02-02] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-03-04] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-04-05] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-05-04] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-06-03] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-07-02] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-08-03] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-09-01] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-10-01] 開始更新，訓練期1年...
 -> 找到 20 對協整配對。
[2021-11-01] 開始

,CAGR(%),總報酬(%),年化波動(%),Sharpe,Sortino,最大回撤(%),日勝率(%)
0,1303.91,3844081.68,268.06,1.66,11.25,-32.83,39.13



【SPY 對標績效指標】


,CAGR(%),總報酬(%),年化波動(%),Sharpe,Sortino,最大回撤(%),日勝率(%)
0,7.19,31.96,75.84,0.38,0.76,-60.39,48.76
